# Optimización de dieta: costo mínimo y cobertura con presupuesto

Dos experimentos de programación lineal (LP) para **una persona y un día**, con el
catálogo CBA–INCAP y los requerimientos calculados por el proyecto:

1. **Costo mínimo:** ¿cuánto cuesta satisfacer todos los requisitos modelados?
2. **Cobertura máxima:** ¿qué proporción de esos requisitos puede alcanzarse con un presupuesto dado?


In [6]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from canasta_inteligente.application.cba_pipeline import load_catalog_pickle

catalog_path = PROJECT_ROOT / 'data' / 'processed' / 'cba_catalog.pkl'
if not catalog_path.exists():
    raise FileNotFoundError(
        f'No existe {catalog_path}. Ejecuta primero cba_pipeline.ipynb '
        'o el módulo canasta_inteligente.application.cba_pipeline.'
    )
catalog = load_catalog_pickle(catalog_path)
print(f'Catálogo: {catalog_path.resolve()}')
print(f'Alimentos disponibles: {len(catalog)}')
print(f'Con información nutricional: {sum(food.nutrition is not None for food in catalog)}')

Catálogo: C:\Users\lp109\Projects\canasta-inteligente-gt\data\processed\cba_catalog.pkl
Alimentos disponibles: 74
Con información nutricional: 74


## Perfil y requerimientos diarios

Se conserva el perfil de referencia del notebook: hombre de 25 años, 65 kg, 1.70 m
y actividad física baja. Modificarlo antes de ejecutar los modelos para evaluar otra persona.

In [24]:
from canasta_inteligente.domain.persona import Persona
from canasta_inteligente.nutrition.evaluation import Peso
from canasta_inteligente.nutrition.service import evaluacion_de_requerimientos_diarios

hombre_promedio = Persona(
    nombre='Hombre promedio de 25 años', edad=25, sexo='hombre',
    peso=78, altura=1.72, naf='moderada',
)
requerimientos = evaluacion_de_requerimientos_diarios(hombre_promedio)
evaluacion_peso = Peso.evaluacion_peso(hombre_promedio)
print(f'Perfil: {hombre_promedio.nombre}')
print(f'IMC: {evaluacion_peso["valor_de_indicador"]:.2f} kg/m²')
print(f'Estado antropométrico: {evaluacion_peso["estado_de_indicador"]}')
print(f'Peso usado en los cálculos: {evaluacion_peso["peso_para_calculos"]:.1f} kg')

Perfil: Hombre promedio de 25 años
IMC: 26.37 kg/m²
Estado antropométrico: sobrepeso
Peso usado en los cálculos: 65.1 kg


In [25]:
requerimientos

{'energia': {'ree': 3678.8000339200003, 'unidad': 'kCal'},
 'proteina': {'rpe': 42.95596799999999,
  'rdd_referencia': 54.020383999999986,
  'rdd_dieta_mixta': 72.89497599999999,
  'unidad': 'g'},
 'carbohidratos': {'ia': None,
  'rpe': 100,
  'rdd_min': 505.83500466400005,
  'rdd_max': 643.7900059360001,
  'unidad': 'g'},
 'azucar': {'ia': None, 'rpe': None, 'rdd': 91.97000084800001, 'unidad': 'g'},
 'fibra': {'ia': None, 'rpe': None, 'rdd': 44.14560040704001, 'unidad': 'g'},
 'lipidos': {'total_min': 81.7511118648889,
  'total_max': 122.62666779733334,
  'saturados_max': 40.87555593244445,
  'poliinsaturados_min': 24.525333559466667,
  'poliinsaturados_max': 44.963111525688895,
  'trans_max': 4.087555593244445,
  'unidad': 'g'},
 'colesterol': {'maximo': 300, 'unidad': 'mg'},
 'micronutrientes': {'vitamina_a': {'rpe': 525,
   'rdd': 750,
   'ia': None,
   'unidad': 'mcg EAR'},
  'retinol': {'imt': 3000, 'unidad': 'mcg'},
  'tiamina': {'ia': None, 'rpe': 1, 'rdd': 1.2, 'unidad': 'mg'}

In [9]:
requerimientos["minerales"]

{'calcio': {'ia': 1000, 'imt': 2500, 'rpe': None, 'rdd': None, 'unidad': 'mg'},
 'fosforo': {'ia': None, 'rpe': 580, 'rdd': 700, 'unidad': 'mg'},
 'magnesio': {'ia': None,
  'rpe': 260.33919999999995,
  'rdd': 312.40703999999994,
  'unidad': 'mg'},
 'hierro': {'alta_biodisponibilidad': {'ia': None, 'rpe': 5.7, 'rdd': 7.5},
  'media_biodisponibilidad': {'ia': None, 'rpe': 8.6, 'rdd': 11.2},
  'baja_biodisponibilidad': {'ia': None, 'rpe': 17.2, 'rdd': 22.4},
  'unidad': 'mg'},
 'zinc': {'alta_biodisponibilidad': {'ia': None, 'rpe': 8.8, 'rdd': 10.6},
  'baja_biodisponibilidad': {'ia': None, 'rpe': 17.7, 'rdd': 21.2},
  'unidad': 'mg',
  'imt': 40},
 'cobre': {'ia': None, 'rpe': 700, 'rdd': 900, 'unidad': 'mcg', 'imt': 10000},
 'selenio': {'ia': None, 'rpe': 45, 'rdd': 54, 'unidad': 'mcg', 'imt': 400}}

# Minimizaccion de costo para total cobertura de requerimientos nutricionales


In [10]:
from pulp import *

In [11]:
from pulp import *
import pandas as pd
from copy import deepcopy
from math import isfinite


def construir_restricciones_base(requerimientos):
    """Traduce los requisitos de este perfil a nutrientes del catalogo."""
    escalar = lambda valor, divisor: None if valor is None else valor / divisor
    get_min_req = lambda nutrient, subrequerimientos: subrequerimientos[nutrient].get('rdd') if subrequerimientos[nutrient].get('rdd') is not None else subrequerimientos[nutrient].get('ia') 
    # ct = close to 
    constraints_keys_base = {
        'energia_kcal': {"ct": requerimientos['energia']['ree']},
        'proteina_g': {"min": requerimientos['proteina']['rdd_dieta_mixta']},
        'carbohidratos_g': {"min": requerimientos['carbohidratos']['rdd_min']},
        'azucares_g': {"max": requerimientos["azucar"]["rdd"]},
        'fibra_dietetica_g': {"min": requerimientos["fibra"]['rdd']},
        'grasa_total_g': {"min": requerimientos["lipidos"]["total_min"], 'max': requerimientos['lipidos']['total_max']},
        'ag_sat_g': {"max": requerimientos["lipidos"]['saturados_max']},
        'ag_poli_g': {"min": requerimientos["lipidos"]["poliinsaturados_min"], 'max': requerimientos["lipidos"]["poliinsaturados_max"]},
        'colesterol_mg': {"ct": requerimientos['colesterol']["maximo"]},
        'vitamina_a_rae_mcg': {"min": get_min_req('vitamina_a', requerimientos["micronutrientes"])},
        'retinol_mcg': {"max": requerimientos['micronutrientes']['retinol']['imt']},
        'tiamina_mg': {"min": get_min_req('tiamina', requerimientos["micronutrientes"]) },
        'riboflavina_mg': {"min": get_min_req('riboflavina', requerimientos["micronutrientes"]) },
        'niacina_mg': {"min": get_min_req('niacina', requerimientos["micronutrientes"]) },
        'vitamina_b6_mg': {"min": get_min_req('vitamina_b6', requerimientos["micronutrientes"]), 'max': requerimientos['micronutrientes']['vitamina_b6']['imt']},
        'folato_fde_mcg': {"min": get_min_req('folatos', requerimientos["micronutrientes"])},
        'ac_folico_mcg': {"max": requerimientos['micronutrientes']["folato_sintetico"]["imt"]},
        'vitamina_b12_mcg': {'min': get_min_req('vitamina_b12', requerimientos["micronutrientes"])},
        'ac_pantotenico_mg': {'min': get_min_req('acido_pantotenico', requerimientos["micronutrientes"])},
        'vitamina_c_mg': {"min": get_min_req('vitamina_c', requerimientos["micronutrientes"])},
        'vitamina_d_mcg': {"min": get_min_req('vitamina_d', requerimientos["micronutrientes"]), "max": requerimientos['micronutrientes']['vitamina_d']['imt']},
        'vitamina_e_mg': {"min": get_min_req('vitamina_e', requerimientos["micronutrientes"]), "max": requerimientos['micronutrientes']['vitamina_e']['imt']},
        'vitamina_k_mcg': {"min": get_min_req("vitamina_k", requerimientos["micronutrientes"])},
        'calcio_mg': {"min": get_min_req("calcio", requerimientos["minerales"]), "max":  requerimientos['minerales']['calcio']['imt']},
        'magnesio_mg': {"min": get_min_req("magnesio", requerimientos["minerales"])},
        'fosforo_mg': {"min": get_min_req("fosforo", requerimientos["minerales"])},
        'selenio_mcg': {"min": get_min_req("selenio", requerimientos["minerales"]), "max":  requerimientos['minerales']['selenio']['imt']},
        'cobre_mg': {"min": escalar(get_min_req("cobre", requerimientos["minerales"]), 1000), "max":  escalar(requerimientos['minerales']['cobre']['imt'], 1000)},
        'zinc_mg':{"min": get_min_req("baja_biodisponibilidad", requerimientos['minerales']['zinc'])}, # Asumimos baja biodisponibildiad  dados los multiples factores que afecta la aborcion del zinc. 
        'potasio_mg': {"min": get_min_req("potasio", requerimientos["electrolitos"])},
        'sodio_mg': {"min": get_min_req("sodio", requerimientos["electrolitos"]), "max":  requerimientos["electrolitos"]['sodio']['limite_sugerido']},
    }



    hierro_constraints = {
        'baja_disponiblidad': {
            "hierro_mg": {'min': get_min_req("baja_biodisponibilidad", requerimientos['minerales']['hierro'])},
            "vitamina_c_mg": {'min': 0.00},
            'carne_g': {'min': 0.00}
        },# Sin restricciones
        'media_disponiblidad': {
            "hierro_mg": {'min': get_min_req("media_biodisponibilidad", requerimientos['minerales']['hierro'])},
            "vitamina_c_mg": { 'min': 25, 'max': 75 },
            'carne_g': {'min': 0.30, 'max': 0.90}
        
        },

        'alta_disponiblidad_A': {
            "hierro_mg": {'min': get_min_req("alta_biodisponibilidad", requerimientos['minerales']['hierro'])},
            "vitamina_c_mg": { 'min': 25, 'max': 75 },
            'carne_g': {'min': 0.3}

        },
        'alta_disponiblidad_B': {
            "hierro_mg": {'min': get_min_req("alta_biodisponibilidad", requerimientos['minerales']['hierro'])},
            "vitamina_c_mg": {'min': 25},
            'carne_g': {'min': 0.9}

        },
    }
    # None significa que el evaluador no proporciona ese limite para este perfil.
    for limites in constraints_keys_base.values():
        for tipo, valor in list(limites.items()):
            if valor is None:
                del limites[tipo]
    return constraints_keys_base, hierro_constraints


def combinar_escenarios(base, hierro, maximizar=False):
    """Combina las condiciones de hierro sin modificar las entradas."""
    escenarios = {}
    for nombre, condiciones in hierro.items():
        restricciones = deepcopy(base)
        if maximizar:
            for limites in restricciones.values():
                if 'min' in limites:
                    limites['meta'] = limites.pop('min')
        restricciones['hierro_mg'] = {
            'meta' if maximizar else 'min': condiciones['hierro_mg']['min'],
        }
        restricciones['carne_g'] = deepcopy(condiciones['carne_g'])
        vitamina_c = restricciones['vitamina_c_mg']
        for tipo, valor in condiciones['vitamina_c_mg'].items():
            combinar = max if tipo == 'min' else min
            vitamina_c[tipo] = combinar(vitamina_c.get(tipo, valor), valor)
        # Conservar los escenarios incompatibles para reportarlos como Infeasible.
        escenarios[f'hierro_{nombre}'] = restricciones
    return escenarios


def construir_escenarios(requerimientos, maximizar=False):
    base, hierro = construir_restricciones_base(requerimientos)
    return combinar_escenarios(base, hierro, maximizar=maximizar)


constraints_keys_base, hierro_constraints = construir_restricciones_base(requerimientos)
possible_constraints = combinar_escenarios(constraints_keys_base, hierro_constraints)


In [12]:
possible_constraints

{'hierro_baja_disponiblidad': {'energia_kcal': {'ct': 2591.88184208},
  'proteina_g': {'min': 72.89497599999999},
  'carbohidratos_g': {'min': 356.383753286},
  'azucares_g': {'max': 64.797046052},
  'fibra_dietetica_g': {'min': 31.10258210496},
  'grasa_total_g': {'min': 57.59737426844444, 'max': 86.39606140266666},
  'ag_sat_g': {'max': 28.79868713422222},
  'ag_poli_g': {'min': 17.279212280533333, 'max': 31.678555847644443},
  'colesterol_mg': {'ct': 300},
  'vitamina_a_rae_mcg': {'min': 750},
  'retinol_mcg': {'max': 3000},
  'tiamina_mg': {'min': 1.2},
  'riboflavina_mg': {'min': 1.3},
  'niacina_mg': {'min': 16},
  'vitamina_b6_mg': {'min': 1.3, 'max': 100},
  'folato_fde_mcg': {'min': 400},
  'ac_folico_mcg': {'max': 1000},
  'vitamina_b12_mcg': {'min': 2.4},
  'ac_pantotenico_mg': {'min': 5.0},
  'vitamina_c_mg': {'min': 75},
  'vitamina_d_mcg': {'min': 5, 'max': 50},
  'vitamina_e_mg': {'min': 15, 'max': 1000},
  'vitamina_k_mcg': {'min': 65.08479999999999},
  'calcio_mg': {'m

In [13]:
catalog.get('arroz').price_summary(region='general')['latest'].cost_per_gram * 100

1.8124999999999996

In [14]:
catalog_variables = [[_food_item, LpVariable(name=f'no_portions_{_food_item.id}', lowBound=0)] for _food_item in catalog]

for _cat_item in catalog_variables:
    aporte = _cat_item[0].nutrition.values_per_100g['fraccion_comestible_pct']
    print(aporte)

1.0
1.0
1.0
1.0
1.0
1.0
0.95
0.56
0.75
0.68
1.0
1.0
1.0
1.0
1.0
0.88
1.0
0.67
0.64
0.65
0.51
0.52
0.91
0.77
0.96
0.75
0.82
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.68
0.98
0.51
1.0
0.6
0.73
0.89
0.8
0.85
0.82
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.67
1.0
1.0
0.56
1.0
1.0
0.62
0.82
0.89
1.0
1.0
1.0
1.0
1.0


In [15]:
from copy import deepcopy
get_latest_price_any_region = lambda _food_item : _food_item.price_summary(region='general')['latest'].cost_per_gram if _food_item.price_summary(region='general')['latest'] is not None else (_food_item.price_summary(region='urbana')['latest'].cost_per_gram if _food_item.price_summary(region='urbana')['latest'] is not None else  _food_item.price_summary(region='rural')['latest'].cost_per_gram)


def extraer_resultado(escenario, modelo, catalog_variables, restricciones,
                     desviaciones, penalizaciones):
    """Guarda una instantánea numérica y las referencias al modelo resuelto."""
    es_optimo = modelo.status == LpStatusOptimal
    filas = []
    if es_optimo:
        for food, variable in catalog_variables:
            porciones = variable.varValue
            gramos = porciones * 100
            precio = get_latest_price_any_region(food)
            filas.append({
                'alimento_id': food.id, 'alimento': food.name,
                'porciones_100g': porciones, 'gramos': gramos,
                'precio_q_por_gramo': precio, 'costo_q': gramos * precio,
            })

    alimentos = pd.DataFrame(filas, columns=[
        'alimento_id', 'alimento', 'porciones_100g', 'gramos',
        'precio_q_por_gramo', 'costo_q',
    ])
    aportes = {}
    if es_optimo:
        for nutriente in restricciones:
            # El modelo usa porciones de 100 g para carne_g; aquí mostramos gramos.
            aportes[nutriente] = sum(
                variable.varValue * (100 * food.is_meat if nutriente == 'carne_g'
                                     else food[nutriente])
                * food.nutrition.values_per_100g['fraccion_comestible_pct']
                for food, variable in catalog_variables
            )

    diagnostico = pd.DataFrame([
        {
            'restriccion': nombre,
            'expresion': str(restriccion),
            'holgura_solver': restriccion.slack if es_optimo else None,
            'precio_sombra': restriccion.pi if es_optimo else None,
        }
        for nombre, restriccion in modelo.constraints.items()
    ])
    return {
        'escenario': escenario,
        'estado': LpStatus[modelo.status],
        'codigo_estado': modelo.status,
        'costo_total_q': float(alimentos['costo_q'].sum()) if es_optimo else None,
        'valor_objetivo': value(modelo.objective) if es_optimo else None,
        'desviaciones': {
            nombre: variable.varValue if es_optimo else None
            for nombre, variable in desviaciones.items()
        },
        'penalizaciones': penalizaciones.copy(),
        'alimentos': alimentos,
        'aportes': pd.Series(aportes, dtype=float, name='aporte_total'),
        'restricciones': deepcopy(restricciones),
        'diagnostico_restricciones': diagnostico,
        'modelo': modelo,
        'variables': {food.id: variable for food, variable in catalog_variables},
    }


def mostrar_resultado(resultado, tolerancia=1e-6):
    """Muestra un resultado guardado sin volver a resolver el problema."""
    print(f"Escenario: {resultado['escenario']}")
    print(f"Estado: {resultado['estado']}")
    if resultado['estado'] != 'Optimal':
        print('No se encontró una solución óptima; no se reportan cantidades ni costos.')
        return

    alimentos = resultado['alimentos']
    display(alimentos.loc[alimentos['porciones_100g'] > tolerancia].round(4))
    print(f"Costo total: Q {resultado['costo_total_q']:.2f}")
    if 'energia_deficit_kcal' in resultado['desviaciones']:
        print(f"Energía por debajo: {resultado['desviaciones']['energia_deficit_kcal']:.2f} kcal")
    if 'energia_exceso_kcal' in resultado['desviaciones']:
        print(f"Energía por encima: {resultado['desviaciones']['energia_exceso_kcal']:.2f} kcal")
    if 'colesterol_exceso_mg' in resultado['desviaciones']:
        print(f"Exceso colesterol: {resultado['desviaciones']['colesterol_exceso_mg']:.2f} mg")
    print(f"Valor función objetivo: {resultado['valor_objetivo']:.4f}")
    if 'coberturas' in resultado:
        print(f"Presupuesto diario: Q {resultado['presupuesto_q']:.2f}")
        print(f"Cobertura media: {resultado['cobertura_media_pct']:.2f} %")
        print(f"Cobertura ponderada: {resultado['cobertura_ponderada_pct']:.2f} %")
        print(f"Penalizacion total: {resultado['penalizacion_total']:.4f}")
        display(resultado['coberturas'].round(4))


In [16]:
from math import isfinite


def minimizar_costo(catalog, escenarios, penalizaciones=None):
    """Minimiza costo mas penalizaciones, con minimos y maximos obligatorios."""
    coeficientes = {'energia': 1.0, 'colesterol': 0.1}
    configuradas = dict(penalizaciones or {})
    desconocidas = set(configuradas) - set(coeficientes)
    if desconocidas:
        raise ValueError(f'Penalizaciones desconocidas: {sorted(desconocidas)}')
    coeficientes.update(configuradas)
    for nombre, coeficiente in coeficientes.items():
        try:
            coeficiente = float(coeficiente)
        except (TypeError, ValueError):
            raise ValueError(f'Penalizacion invalida para {nombre}: debe ser numerica.') from None
        if not isfinite(coeficiente) or coeficiente < 0:
            raise ValueError(f'Penalizacion invalida para {nombre}: debe ser finita y no negativa.')
        coeficientes[nombre] = coeficiente

    resultados = []
    for escenario, restricciones in escenarios.items():
        modelo = LpProblem(f"MinimizeCost_{escenario}", LpMinimize)
        variables = [
            (food, LpVariable(f'no_portions_{food.id}', lowBound=0))
            for food in catalog
        ]
        aportes = {}
        for nutriente in restricciones:
            if nutriente == 'carne_g':
                aportes[nutriente] = lpSum(
                    variable * food.nutrition.values_per_100g['fraccion_comestible_pct']
                    for food, variable in variables if food.is_meat
                )
            else:
                aportes[nutriente] = lpSum(
                    variable * food[nutriente]
                    * food.nutrition.values_per_100g['fraccion_comestible_pct']
                    for food, variable in variables
                )

        desviaciones = {}
        terminos_penalizacion = []
        for nutriente, limites in restricciones.items():
            if 'ct' in limites:
                referencia = limites['ct']
                if not isfinite(referencia) or referencia < 0:
                    raise ValueError(f'Referencia ct invalida para {nutriente}.')
                if nutriente == 'energia_kcal':
                    deficit = LpVariable('deEminus', lowBound=0)
                    exceso = LpVariable('deEplus', lowBound=0)
                    modelo += aportes[nutriente] + deficit - exceso == referencia, 'REE'
                    desviaciones['energia_deficit_kcal'] = deficit
                    desviaciones['energia_exceso_kcal'] = exceso
                    terminos_penalizacion.append(coeficientes['energia'] * (deficit + exceso))
                elif nutriente == 'colesterol_mg':
                    exceso = LpVariable('deCPlus', lowBound=0)
                    modelo += aportes[nutriente] - exceso <= referencia, 'Colesterol'
                    desviaciones['colesterol_exceso_mg'] = exceso
                    terminos_penalizacion.append(coeficientes['colesterol'] * exceso)
                else:
                    raise ValueError(f'Nutriente con referencia ct no admitido: {nutriente}')
            if 'min' in limites:
                modelo += aportes[nutriente] >= limites['min'], f'limite_inferior_de_{nutriente}'
            if 'max' in limites:
                modelo += aportes[nutriente] <= limites['max'], f'limite_superior_de_{nutriente}'

        costo = lpSum(
            variable * get_latest_price_any_region(food) * 100
            for food, variable in variables
        )
        modelo += costo + lpSum(terminos_penalizacion), 'costo_mas_penalizaciones'
        modelo.solve(PULP_CBC_CMD(msg=0))

        resultado = extraer_resultado(
            escenario, modelo, variables, restricciones,
            desviaciones=desviaciones, penalizaciones=coeficientes,
        )
        # Reportar desviaciones reales incluso si un coeficiente es cero:
        # en ese caso el solver no esta obligado a minimizar las holguras.
        if modelo.status == LpStatusOptimal:
            if 'energia_deficit_kcal' in desviaciones:
                diferencia = resultado['aportes']['energia_kcal'] - restricciones['energia_kcal']['ct']
                resultado['desviaciones']['energia_deficit_kcal'] = max(0.0, -diferencia)
                resultado['desviaciones']['energia_exceso_kcal'] = max(0.0, diferencia)
            if 'colesterol_exceso_mg' in desviaciones:
                resultado['desviaciones']['colesterol_exceso_mg'] = max(
                    0.0, resultado['aportes']['colesterol_mg'] - restricciones['colesterol_mg']['ct'],
                )
        resultado['penalizacion_total'] = (
            coeficientes['energia'] * (
                resultado['desviaciones'].get('energia_deficit_kcal', 0.0)
                + resultado['desviaciones'].get('energia_exceso_kcal', 0.0)
            ) + coeficientes['colesterol'] * resultado['desviaciones'].get('colesterol_exceso_mg', 0.0)
            if modelo.status == LpStatusOptimal else None
        )
        resultados.append(resultado)
    return resultados


lambda_co = 0.1
lambda_ree = 1
penalizaciones_minimizacion = {'energia': lambda_ree, 'colesterol': lambda_co}
catalog_variable_results = minimizar_costo(catalog, possible_constraints, penalizaciones_minimizacion)
lps = [resultado['modelo'] for resultado in catalog_variable_results]
for resultado_minimo in catalog_variable_results:
    mostrar_resultado(resultado_minimo)


Escenario: hierro_baja_disponiblidad
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
11,leche_en_polvo,LECHE EN POLVO,0.3737,37.3666,0.0983,3.6741
15,huevos_de_gallina,HUEVOS DE GALLINA,0.1863,18.6342,0.0273,0.5090
16,aceites_comestibles,ACEITES COMESTIBLES,0.3081,30.8062,0.0253,0.7785
27,frijol,FRIJÓL,3.7513,375.1260,0.0165,6.1970
29,sal,SAL,0.0233,2.3330,0.0025,0.0058
34,maiz_blanco,Maíz blanco,1.7938,179.3839,0.0050,0.8890
61,cereales_de_bolsa_o_caja,Cereales de bolsa o caja,0.2079,20.7928,0.0490,1.0188
67,lechuga_achicoria_y_rugula_fresca_o_refrigerada,"Lechuga, achicoria y rúgula, fresca o refrigerada",5.4404,544.0398,0.0101,5.4861


Costo total: Q 18.56
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 18.5585
Escenario: hierro_media_disponiblidad
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
11,leche_en_polvo,LECHE EN POLVO,0.1321,13.2136,0.0983,1.2992
15,huevos_de_gallina,HUEVOS DE GALLINA,0.8340,83.4012,0.0273,2.2781
16,aceites_comestibles,ACEITES COMESTIBLES,0.0033,0.3314,0.0253,0.0084
27,frijol,FRIJÓL,4.2560,425.6036,0.0165,7.0309
29,sal,SAL,0.0152,1.5177,0.0025,0.0038
34,maiz_blanco,Maíz blanco,1.0167,101.6686,0.0050,0.5039
41,pescado_entero_fresco_refrigerado_o_congelado,"Pescado entero, fresco, refrigerado o congelado",0.3482,34.8189,0.0551,1.9173
42,margarina_vegetal_regular,Margarina vegetal regular,0.6431,64.3070,0.0389,2.4999
61,cereales_de_bolsa_o_caja,Cereales de bolsa o caja,0.0892,8.9219,0.0490,0.4372
62,carne_de_res_molida,Carne de res molida,0.1224,12.2424,0.0754,0.9236


Costo total: Q 20.57
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 20.5701
Escenario: hierro_alta_disponiblidad_A
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
11,leche_en_polvo,LECHE EN POLVO,0.1321,13.2136,0.0983,1.2992
15,huevos_de_gallina,HUEVOS DE GALLINA,0.8340,83.4012,0.0273,2.2781
16,aceites_comestibles,ACEITES COMESTIBLES,0.0033,0.3314,0.0253,0.0084
27,frijol,FRIJÓL,4.2560,425.6036,0.0165,7.0309
29,sal,SAL,0.0152,1.5177,0.0025,0.0038
34,maiz_blanco,Maíz blanco,1.0167,101.6686,0.0050,0.5039
41,pescado_entero_fresco_refrigerado_o_congelado,"Pescado entero, fresco, refrigerado o congelado",0.3482,34.8189,0.0551,1.9173
42,margarina_vegetal_regular,Margarina vegetal regular,0.6431,64.3070,0.0389,2.4999
61,cereales_de_bolsa_o_caja,Cereales de bolsa o caja,0.0892,8.9219,0.0490,0.4372
62,carne_de_res_molida,Carne de res molida,0.1224,12.2424,0.0754,0.9236


Costo total: Q 20.57
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 20.5701
Escenario: hierro_alta_disponiblidad_B
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
10,embutidos,EMBUTIDOS,0.1788,17.8845,0.0442,0.7900
11,leche_en_polvo,LECHE EN POLVO,0.3876,38.7619,0.0983,3.8113
16,aceites_comestibles,ACEITES COMESTIBLES,0.0729,7.2864,0.0253,0.1841
27,frijol,FRIJÓL,2.1593,215.9271,0.0165,3.5671
29,sal,SAL,0.0101,1.0078,0.0025,0.0025
34,maiz_blanco,Maíz blanco,3.4217,342.1699,0.0050,1.6958
40,longanizas_y_chorizos_de_todo_tipo,Longanizas y chorizos (de todo tipo),0.5470,54.7037,0.0490,2.6794
62,carne_de_res_molida,Carne de res molida,0.1851,18.5059,0.0754,1.3961
67,lechuga_achicoria_y_rugula_fresca_o_refrigerada,"Lechuga, achicoria y rúgula, fresca o refrigerada",6.1338,613.3779,0.0101,6.1853


Costo total: Q 20.31
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 20.3116


## Explorar las soluciones

`catalog_variable_results` conserva un diccionario por escenario, incluso si no es
óptimo. Incluye alimentos (porciones de **100 g**), costo en quetzales, aportes,
desviaciones, restricciones y el modelo PuLP. Las tablas son instantáneas de esta
ejecución; `modelo` y `variables` permiten inspeccionar los objetos del solver.

El costo de alimentos y el objetivo son distintos: el objetivo suma también las
penalizaciones de energía y colesterol. Los escenarios sin solución óptima tienen
sus métricas en `None`, para no interpretar valores del solver como una dieta válida.

In [17]:
# Comparación de todos los escenarios resueltos.
resultados_por_escenario = {r['escenario']: r for r in catalog_variable_results}
resumen_soluciones = pd.DataFrame([
    {
        'escenario': r['escenario'], 'estado': r['estado'],
        'costo_total_q': r['costo_total_q'], 'valor_objetivo': r['valor_objetivo'],
        'alimentos_seleccionados': int((r['alimentos']['porciones_100g'] > 1e-6).sum())
        if r['estado'] == 'Optimal' else None,
        **r['desviaciones'],
    }
    for r in catalog_variable_results
])
display(resumen_soluciones)

,escenario,estado,costo_total_q,valor_objetivo,alimentos_seleccionados,energia_deficit_kcal,energia_exceso_kcal,colesterol_exceso_mg
0,hierro_baja_disponiblidad,Optimal,18.558460,18.558460,8,0.000002,0.000000,0.0
1,hierro_media_disponiblidad,Optimal,20.570146,20.570146,11,0.000000,0.000007,0.0
2,hierro_alta_disponiblidad_A,Optimal,20.570146,20.570146,11,0.000000,0.000007,0.0
3,hierro_alta_disponiblidad_B,Optimal,20.311609,20.311609,9,0.000000,0.000021,0.0


In [18]:
# Cambia este nombre para explorar otro escenario disponible.
escenario_elegido = next(
    (r['escenario'] for r in catalog_variable_results if r['estado'] == 'Optimal'),
    next(iter(resultados_por_escenario), None),
)
# Ejemplo: escenario_elegido = 'hierro_alta_disponiblidad_B'
solucion = resultados_por_escenario.get(escenario_elegido)
if solucion is None:
    print('No hay escenarios resueltos. Revisa possible_constraints.')
else:
    mostrar_resultado(solucion)

Escenario: hierro_baja_disponiblidad
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
11,leche_en_polvo,LECHE EN POLVO,0.3737,37.3666,0.0983,3.6741
15,huevos_de_gallina,HUEVOS DE GALLINA,0.1863,18.6342,0.0273,0.5090
16,aceites_comestibles,ACEITES COMESTIBLES,0.3081,30.8062,0.0253,0.7785
27,frijol,FRIJÓL,3.7513,375.1260,0.0165,6.1970
29,sal,SAL,0.0233,2.3330,0.0025,0.0058
34,maiz_blanco,Maíz blanco,1.7938,179.3839,0.0050,0.8890
61,cereales_de_bolsa_o_caja,Cereales de bolsa o caja,0.2079,20.7928,0.0490,1.0188
67,lechuga_achicoria_y_rugula_fresca_o_refrigerada,"Lechuga, achicoria y rúgula, fresca o refrigerada",5.4404,544.0398,0.0101,5.4861


Costo total: Q 18.56
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 18.5585


In [19]:
# Aportes totales frente a los límites y referencias del escenario elegido.
# carne_g se expresa aquí en gramos; sus límites originales son porciones de 100 g.
# ct es una referencia con penalización, no una garantía de cumplimiento exacto.
filas_nutrientes = []
if solucion is not None and solucion['estado'] == 'Optimal':
    for nutriente, limites in solucion['restricciones'].items():
        factor = 100 if nutriente == 'carne_g' else 1
        aporte = solucion['aportes'][nutriente]
        minimo = limites.get('min')
        filas_nutrientes.append({
            'nutriente': nutriente,
            'aporte_total': aporte,
            'minimo': minimo * factor if minimo is not None else None,
            'maximo': limites['max'] * factor if 'max' in limites else None,
            'referencia_ct': limites['ct'] * factor if 'ct' in limites else None,
            'cobertura_min_pct': 100 * aporte / (minimo * factor)
            if minimo is not None and minimo > 0 else None,
        })
    tabla_nutrientes = pd.DataFrame(filas_nutrientes).set_index('nutriente')
    display(tabla_nutrientes.round(3))
else:
    tabla_nutrientes = pd.DataFrame()
    print('Selecciona un escenario con solución óptima para consultar aportes.')

,aporte_total,minimo,maximo,referencia_ct,cobertura_min_pct
nutriente,,,,,
energia_kcal,2591.882,NaN,NaN,2591.882,NaN
proteina_g,118.963,72.895,NaN,NaN,163.198
carbohidratos_g,420.067,356.384,NaN,NaN,117.869
azucares_g,28.616,NaN,64.797,NaN,NaN
fibra_dietetica_g,89.770,31.103,NaN,NaN,288.626
grasa_total_g,57.597,57.597,86.396,NaN,100.000
ag_sat_g,11.984,NaN,28.799,NaN,NaN
ag_poli_g,27.415,17.279,31.679,NaN,158.660
colesterol_mg,97.247,NaN,NaN,300.000,NaN


In [20]:
# Restricciones efectivamente incluidas en el LP.
# Una holgura cercana a cero indica una restricción activa.
# Se conserva el signo de CBC: puede ser negativo para restricciones >=.
if solucion is not None:
    display(solucion['diagnostico_restricciones'])
    # Para consultar el modelo completo: print(solucion['modelo'])

,restriccion,expresion,holgura_solver,precio_sombra
0,REE,deEminus - deEplus + 884.0*no_portions_aceites...,4.208000e-05,-0.005413
1,limite_inferior_de_proteina_g,1.4941*no_portions_aguacates + 0.07*no_portion...,-4.606808e+01,0.000000
2,limite_inferior_de_carbohidratos_g,5.239400000000001*no_portions_aguacates + 9.56...,-6.368372e+01,0.000000
3,limite_superior_de_azucares_g,1.6214*no_portions_aguacates + 8.97*no_portion...,3.618128e+01,0.000000
4,limite_inferior_de_fibra_dietetica_g,3.752*no_portions_aguacates + 1.42400000000000...,-5.866765e+01,0.000000
5,limite_inferior_de_grasa_total_g,100.0*no_portions_aceites_comestibles + 6.7402...,2.684444e-07,0.034104
6,limite_superior_de_grasa_total_g,100.0*no_portions_aceites_comestibles + 6.7402...,2.879869e+01,0.000000
7,limite_superior_de_ag_sat_g,7.43*no_portions_aceites_comestibles + 1.31320...,1.681428e+01,0.000000
8,limite_inferior_de_ag_poli_g,65.14*no_portions_aceites_comestibles + 1.1256...,-1.013599e+01,0.000000
9,limite_superior_de_ag_poli_g,65.14*no_portions_aceites_comestibles + 1.1256...,4.263350e+00,0.000000


In [21]:
# Cantidades por alimento para comparar las dietas de los escenarios óptimos.
tablas = [
    r['alimentos'].assign(escenario=r['escenario'])
    for r in catalog_variable_results if r['estado'] == 'Optimal'
]
if tablas:
    gramos_por_escenario = pd.concat(tablas, ignore_index=True).pivot_table(
        index=['alimento_id', 'alimento'], columns='escenario',
        values='gramos', aggfunc='sum', fill_value=0,
    )
    gramos_por_escenario = gramos_por_escenario.loc[
        (gramos_por_escenario > 1e-4).any(axis=1)
    ]
    display(gramos_por_escenario.round(1))
else:
    gramos_por_escenario = pd.DataFrame()
    print('No hay soluciones óptimas para comparar.')

,escenario,hierro_alta_disponiblidad_A,hierro_alta_disponiblidad_B,hierro_baja_disponiblidad,hierro_media_disponiblidad
alimento_id,alimento,,,,
aceites_comestibles,ACEITES COMESTIBLES,0.3,7.3,30.8,0.3
carne_de_res_molida,Carne de res molida,12.2,18.5,0.0,12.2
cereales_de_bolsa_o_caja,Cereales de bolsa o caja,8.9,0.0,20.8,8.9
embutidos,EMBUTIDOS,0.0,17.9,0.0,0.0
frijol,FRIJÓL,425.6,215.9,375.1,425.6
huevos_de_gallina,HUEVOS DE GALLINA,83.4,0.0,18.6,83.4
leche_en_polvo,LECHE EN POLVO,13.2,38.8,37.4,13.2
lechuga_achicoria_y_rugula_fresca_o_refrigerada,"Lechuga, achicoria y rúgula, fresca o refrigerada",363.7,613.4,544.0,363.7
longanizas_y_chorizos_de_todo_tipo,Longanizas y chorizos (de todo tipo),0.0,54.7,0.0,0.0


#  Maximización de cobertura nutricional con restricciones de presupuesto. 

El presupuesto diario es una restricción obligatoria. Se maximiza la suma
ponderada de coberturas nutricionales, con un tope del 100 % por meta: los excesos
no aumentan la puntuación. Las metas pueden cumplirse parcialmente.

Edita `pesos_nutrientes` junto a `daily_budget`: los nutrientes omitidos usan peso
`1`; `0.5` reduce su importancia a la mitad, `2` la duplica y `0` elimina su
contribución al objetivo, pero conserva sus límites y su cobertura en el reporte.
Debe quedar al menos una meta con peso positivo en cada escenario.
La cobertura media trata a todos los nutrientes por igual; la cobertura ponderada
usa los pesos elegidos y divide por su suma. Ambas se muestran en porcentaje.

Los máximos obligatorios y las condiciones de carne y vitamina C de cada escenario de hierro
siguen siendo obligatorios; un presupuesto insuficiente para esas condiciones
puede hacer que el escenario sea infactible. La carne se mide en porciones
comestibles de 100 g en el modelo y en gramos comestibles en el reporte.

La energía usa una referencia `ct`: se penalizan tanto el déficit como el exceso.
El colesterol usa `ct` como umbral flexible: solo se penaliza el exceso, sin un
máximo obligatorio. Ambos se muestran como desviaciones, fuera de las coberturas.

Edita `penalizaciones_cobertura` junto a los pesos. El objetivo es:

`sum(peso * cobertura) - energia * (deficit_kcal + exceso_kcal) - colesterol * exceso_mg`

Los coeficientes iniciales son `energia=1.0` y `colesterol=0.1`, como en la
minimización. Se aplican por kcal y mg, respectivamente; no son porcentajes.
Reducirlos permite intercambiar más desviación por cobertura, y `0` desactiva esa
penalización. Las coberturas porcentuales se reportan antes de penalizaciones;
`valor_objetivo` incluye la resta y puede ser negativo.

Los resultados se guardan en `coverage_results` y `coverage_lps`, separados de
la minimización, con sus propias desviaciones y coeficientes.


In [22]:
# Cada llamada construye restricciones nuevas para los requisitos recibidos.
constraints_keys_base_maximize, hierro_constraints_maximize = construir_restricciones_base(requerimientos)
possible_constraints_maximize = combinar_escenarios(
    constraints_keys_base_maximize, hierro_constraints_maximize, maximizar=True,
)


In [23]:
from math import isfinite


def maximizar_cobertura(catalog, escenarios, presupuesto, pesos_nutrientes=None,
                       penalizaciones=None):
    """Resuelve cada escenario con metas flexibles y limites obligatorios."""
    if not isfinite(presupuesto) or presupuesto < 0:
        raise ValueError('El presupuesto debe ser finito y no negativo.')

    # Los nutrientes omitidos conservan peso 1. Copia para no modificar la entrada.
    pesos_configurados = dict(pesos_nutrientes or {})
    nutrientes_con_meta = {
        nutriente for restricciones in escenarios.values()
        for nutriente, limites in restricciones.items()
        if nutriente != 'carne_g' and limites.get('meta') is not None
        and limites['meta'] > 0
    }
    desconocidos = set(pesos_configurados) - nutrientes_con_meta
    if desconocidos:
        raise ValueError(f'Pesos sin una meta nutricional positiva: {sorted(desconocidos)}')
    for nutriente, peso in pesos_configurados.items():
        try:
            peso = float(peso)
        except (TypeError, ValueError):
            raise ValueError(f'Peso invalido para {nutriente}: debe ser numerico.') from None
        if not isfinite(peso) or peso < 0:
            raise ValueError(f'Peso invalido para {nutriente}: debe ser finito y no negativo.')
        pesos_configurados[nutriente] = peso

    coeficientes = {'energia': 1.0, 'colesterol': 0.1}
    configuradas = dict(penalizaciones or {})
    desconocidas = set(configuradas) - set(coeficientes)
    if desconocidas:
        raise ValueError(f'Penalizaciones desconocidas: {sorted(desconocidas)}')
    coeficientes.update(configuradas)
    for nombre, coeficiente in coeficientes.items():
        try:
            coeficiente = float(coeficiente)
        except (TypeError, ValueError):
            raise ValueError(f'Penalizacion invalida para {nombre}: debe ser numerica.') from None
        if not isfinite(coeficiente) or coeficiente < 0:
            raise ValueError(f'Penalizacion invalida para {nombre}: debe ser finita y no negativa.')
        coeficientes[nombre] = coeficiente

    resultados = []
    for escenario, restricciones in escenarios.items():
        modelo = LpProblem(f"MaximizeCoverage_{escenario}", LpMaximize)
        variables = [
            (food, LpVariable(f'no_portions_{food.id}', lowBound=0))
            for food in catalog
        ]
        aportes = {}
        for nutriente in restricciones:
            if nutriente == 'carne_g':
                aportes[nutriente] = lpSum(
                    variable * food.nutrition.values_per_100g['fraccion_comestible_pct']
                    for food, variable in variables if food.is_meat
                )
            else:
                aportes[nutriente] = lpSum(
                    variable * food[nutriente]
                    * food.nutrition.values_per_100g['fraccion_comestible_pct']
                    for food, variable in variables
                )

        metas = {
            nutriente: limites['meta']
            for nutriente, limites in restricciones.items()
            if nutriente != 'carne_g' and limites.get('meta') is not None
            and limites['meta'] > 0
        }
        if not metas:
            raise ValueError(f'El escenario {escenario} no tiene metas positivas.')
        pesos = {nutriente: pesos_configurados.get(nutriente, 1.0) for nutriente in metas}
        suma_pesos = sum(pesos.values())
        if not isfinite(suma_pesos) or suma_pesos <= 0:
            raise ValueError(f'El escenario {escenario} requiere una suma de pesos positiva y finita.')
        coberturas = {
            nutriente: LpVariable(f'cobertura_{nutriente}', lowBound=0, upBound=1)
            for nutriente in metas
        }
        puntuacion_cobertura = lpSum(
            pesos[nutriente] * cobertura for nutriente, cobertura in coberturas.items()
        )
        desviaciones = {}
        terminos_penalizacion = []
        for nutriente, meta in metas.items():
            modelo += aportes[nutriente] >= meta * coberturas[nutriente], f'cobertura_de_{nutriente}'

        for nutriente, limites in restricciones.items():
            if 'ct' in limites:
                referencia = limites['ct']
                if not isfinite(referencia) or referencia < 0:
                    raise ValueError(f'Referencia ct invalida para {nutriente}.')
                if nutriente == 'energia_kcal':
                    deficit = LpVariable('deEminus', lowBound=0)
                    exceso = LpVariable('deEplus', lowBound=0)
                    modelo += aportes[nutriente] + deficit - exceso == referencia, 'REE'
                    desviaciones['energia_deficit_kcal'] = deficit
                    desviaciones['energia_exceso_kcal'] = exceso
                    terminos_penalizacion.append(coeficientes['energia'] * (deficit + exceso))
                elif nutriente == 'colesterol_mg':
                    exceso = LpVariable('deCPlus', lowBound=0)
                    modelo += aportes[nutriente] - exceso <= referencia, 'Colesterol'
                    desviaciones['colesterol_exceso_mg'] = exceso
                    terminos_penalizacion.append(coeficientes['colesterol'] * exceso)
                else:
                    raise ValueError(f'Nutriente con referencia ct no admitido: {nutriente}')
            if 'min' in limites:
                modelo += aportes[nutriente] >= limites['min'], f'limite_inferior_de_{nutriente}'
            if 'max' in limites:
                modelo += aportes[nutriente] <= limites['max'], f'limite_superior_de_{nutriente}'

        costo = lpSum(
            variable * get_latest_price_any_region(food) * 100
            for food, variable in variables
        )
        modelo += costo <= presupuesto, 'presupuesto_diario'
        modelo += puntuacion_cobertura - lpSum(terminos_penalizacion), 'cobertura_menos_penalizaciones'
        modelo.solve(PULP_CBC_CMD(msg=0))

        resultado = extraer_resultado(
            escenario, modelo, variables, restricciones,
            desviaciones=desviaciones, penalizaciones=coeficientes,
        )
        # Reportar desviaciones reales incluso si un coeficiente es cero:
        # en ese caso el solver no esta obligado a minimizar las holguras.
        if modelo.status == LpStatusOptimal:
            if 'energia_deficit_kcal' in desviaciones:
                diferencia = resultado['aportes']['energia_kcal'] - restricciones['energia_kcal']['ct']
                resultado['desviaciones']['energia_deficit_kcal'] = max(0.0, -diferencia)
                resultado['desviaciones']['energia_exceso_kcal'] = max(0.0, diferencia)
            if 'colesterol_exceso_mg' in desviaciones:
                resultado['desviaciones']['colesterol_exceso_mg'] = max(
                    0.0, resultado['aportes']['colesterol_mg'] - restricciones['colesterol_mg']['ct'],
                )
        resultado['penalizacion_total'] = (
            coeficientes['energia'] * (
                resultado['desviaciones'].get('energia_deficit_kcal', 0.0)
                + resultado['desviaciones'].get('energia_exceso_kcal', 0.0)
            ) + coeficientes['colesterol'] * resultado['desviaciones'].get('colesterol_exceso_mg', 0.0)
            if modelo.status == LpStatusOptimal else None
        )
        resultado['presupuesto_q'] = presupuesto
        resultado['pesos_nutrientes'] = pesos.copy()
        filas = []
        if modelo.status == LpStatusOptimal:
            for nutriente, meta in metas.items():
                aporte = resultado['aportes'][nutriente]
                filas.append({
                    'nutriente': nutriente, 'meta': meta, 'aporte': aporte,
                    'peso': pesos[nutriente],
                    'cobertura_pct': min(1.0, max(0.0, aporte / meta)) * 100,
                    'deficit': max(0.0, meta - aporte),
                })
        resultado['coberturas'] = pd.DataFrame(filas, columns=[
            'nutriente', 'meta', 'aporte', 'peso', 'cobertura_pct', 'deficit',
        ]).set_index('nutriente')
        resultado['cobertura_media_pct'] = (
            float(resultado['coberturas']['cobertura_pct'].mean())
            if modelo.status == LpStatusOptimal else None
        )
        resultado['cobertura_ponderada_pct'] = (
            float((resultado['coberturas']['cobertura_pct']
                   * resultado['coberturas']['peso']).sum() / suma_pesos)
            if modelo.status == LpStatusOptimal else None
        )
        resultados.append(resultado)
    return resultados


daily_budget = 15
# Edita estos pesos segun el caso. Los nutrientes no incluidos tienen peso 1.
# 0 = no puntua; 0.25 = un cuarto de importancia; 1 = normal; 2 = doble.
# Los valores iniciales conservan la ponderacion uniforme anterior.
pesos_nutrientes = {
    'vitamina_d_mcg': 1.0,  # Ejemplo: cambiar a 0.25 para reducir su importancia.
    'fibra_dietetica_g': 1.0,  # Ejemplo: cambiar a 0.5.
}
# Puntos restados del objetivo por kcal de desviacion y por mg de exceso.
# Valores iniciales iguales a los de minimizacion, editables por separado.
penalizaciones_cobertura = {
    'energia': 1.0,
    'colesterol': 0.1,
}
coverage_results = maximizar_cobertura(
    catalog, possible_constraints_maximize, daily_budget, pesos_nutrientes,
    penalizaciones=penalizaciones_cobertura,
)
coverage_lps = [resultado['modelo'] for resultado in coverage_results]
for resultado_cobertura in coverage_results:
    mostrar_resultado(resultado_cobertura)

resumen_coberturas = pd.DataFrame([
    {
        'escenario': r['escenario'], 'estado': r['estado'],
        'presupuesto_q': r['presupuesto_q'], 'costo_total_q': r['costo_total_q'],
        'cobertura_media_pct': r['cobertura_media_pct'],
        'cobertura_ponderada_pct': r['cobertura_ponderada_pct'],
        'energia_deficit_kcal': r['desviaciones'].get('energia_deficit_kcal'),
        'energia_exceso_kcal': r['desviaciones'].get('energia_exceso_kcal'),
        'colesterol_exceso_mg': r['desviaciones'].get('colesterol_exceso_mg'),
        'penalizacion_total': r['penalizacion_total'],
        'valor_objetivo': r['valor_objetivo'],
    }
    for r in coverage_results
])
display(resumen_coberturas)


Escenario: hierro_baja_disponiblidad
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
11,leche_en_polvo,LECHE EN POLVO,0.4010,40.0980,0.0983,3.9427
16,aceites_comestibles,ACEITES COMESTIBLES,0.2152,21.5220,0.0253,0.5439
26,hierbas,HIERBAS,0.3220,32.1982,0.0150,0.4815
27,frijol,FRIJÓL,0.7833,78.3313,0.0165,1.2940
29,sal,SAL,0.0213,2.1305,0.0025,0.0053
34,maiz_blanco,Maíz blanco,4.7814,478.1412,0.0050,2.3696
61,cereales_de_bolsa_o_caja,Cereales de bolsa o caja,0.2194,21.9363,0.0490,1.0749
67,lechuga_achicoria_y_rugula_fresca_o_refrigerada,"Lechuga, achicoria y rúgula, fresca o refrigerada",5.2440,524.4018,0.0101,5.2881


Costo total: Q 15.00
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 25.7972
Presupuesto diario: Q 15.00
Cobertura media: 99.22 %
Cobertura ponderada: 99.22 %
Penalizacion total: 0.0000


,meta,aporte,peso,cobertura_pct,deficit
nutriente,,,,,
proteina_g,72.8950,82.7370,1.0,100.000,0.0000
carbohidratos_g,356.3838,460.3030,1.0,100.000,0.0000
fibra_dietetica_g,31.1026,65.3396,1.0,100.000,0.0000
grasa_total_g,57.5974,57.5974,1.0,100.000,0.0000
ag_poli_g,17.2792,25.7777,1.0,100.000,0.0000
vitamina_a_rae_mcg,750.0000,1449.7446,1.0,100.000,0.0000
tiamina_mg,1.2000,3.2973,1.0,100.000,0.0000
riboflavina_mg,1.3000,2.4488,1.0,100.000,0.0000
niacina_mg,16.0000,25.4931,1.0,100.000,0.0000


Escenario: hierro_media_disponiblidad
Estado: Optimal


...

Escenario: hierro_alta_disponiblidad_B
Estado: Optimal


,alimento_id,alimento,porciones_100g,gramos,precio_q_por_gramo,costo_q
10,embutidos,EMBUTIDOS,0.9000,90.0000,0.0442,3.9756
11,leche_en_polvo,LECHE EN POLVO,0.3799,37.9949,0.0983,3.7359
16,aceites_comestibles,ACEITES COMESTIBLES,0.0608,6.0764,0.0253,0.1536
26,hierbas,HIERBAS,0.1898,18.9765,0.0150,0.2838
27,frijol,FRIJÓL,1.1786,117.8624,0.0165,1.9471
29,sal,SAL,0.0023,0.2338,0.0025,0.0006
34,maiz_blanco,Maíz blanco,4.4242,442.4159,0.0050,2.1926
67,lechuga_achicoria_y_rugula_fresca_o_refrigerada,"Lechuga, achicoria y rúgula, fresca o refrigerada",2.6884,268.8396,0.0101,2.7110


Costo total: Q 15.00
Energía por debajo: 0.00 kcal
Energía por encima: 0.00 kcal
Exceso colesterol: 0.00 mg
Valor función objetivo: 25.1856
Presupuesto diario: Q 15.00
Cobertura media: 96.87 %
Cobertura ponderada: 96.87 %
Penalizacion total: 0.0000


,meta,aporte,peso,cobertura_pct,deficit
nutriente,,,,,
proteina_g,72.8950,92.0056,1.0,100.0000,0.0000
carbohidratos_g,356.3838,429.9130,1.0,100.0000,0.0000
fibra_dietetica_g,31.1026,59.6008,1.0,100.0000,0.0000
grasa_total_g,57.5974,64.5308,1.0,100.0000,0.0000
ag_poli_g,17.2792,17.2792,1.0,100.0000,0.0000
vitamina_a_rae_mcg,750.0000,750.0000,1.0,100.0000,0.0000
tiamina_mg,1.2000,3.2422,1.0,100.0000,0.0000
riboflavina_mg,1.3000,1.9530,1.0,100.0000,0.0000
niacina_mg,16.0000,22.2416,1.0,100.0000,0.0000


,escenario,estado,presupuesto_q,costo_total_q,cobertura_media_pct,cobertura_ponderada_pct,energia_deficit_kcal,energia_exceso_kcal,colesterol_exceso_mg,penalizacion_total,valor_objetivo
0,hierro_baja_disponiblidad,Optimal,15,15.0,99.220078,99.220078,0.000000,0.000020,0.0,0.000020,25.797220
1,hierro_media_disponiblidad,Optimal,15,15.0,98.182000,98.182000,0.000000,0.000008,0.0,0.000008,25.527320
2,hierro_alta_disponiblidad_A,Optimal,15,15.0,98.182000,98.182000,0.000000,0.000008,0.0,0.000008,25.527320
3,hierro_alta_disponiblidad_B,Optimal,15,15.0,96.867720,96.867720,0.000006,0.000000,0.0,0.000006,25.185607
